# DressApp Eyes — Gemma-4 E4B Multi-LoRA Fine-Tuning Notebook

This standalone Google Colab notebook provides an on-demand, interactive fallback to fine-tune specialized LoRA adapters for **DressApp Eyes** (`google/gemma-4-E4B-it`).

### Supported Workflows & Adapters:
1. **`garment_vision`**: Strict JSON garment attribute extraction
2. **`trend_scout`**: Trend analysis & aesthetic classification
3. **`stylist_chat`**: Conversational personal wardrobe styling
4. **`suitcase`**: Capsule packing manifest generator
5. **`scheduled_outfit`**: Weather/calendar outfit scheduling

### Key Technical Features:
- 4-bit NF4 Quantization with double-quantization (`BitsAndBytesConfig`)
- Complete LoRA projections on `['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']`
- **Assistant-Only Loss Masking**: Cross-entropy loss computed strictly on output JSON / assistant responses
- Export to Hugging Face Hub or direct `.zip` download for Hetzner VPS deployment

## 1. Environment Setup & Dependencies

In [ ]:
# Check GPU accelerator
!nvidia-smi

# Install latest PyTorch, HuggingFace, PEFT, TRL and bitsandbytes
!pip install -q -U \
    "torch>=2.2.0" \
    "transformers>=4.40.0" \
    "peft>=0.10.0" \
    "trl>=0.8.6" \
    "accelerate>=0.28.0" \
    "bitsandbytes>=0.43.0" \
    "datasets>=2.18.0" \
    "huggingface_hub>=0.22.0"

## 2. Google Drive Mount / GitHub Repository Clone

In [ ]:
import os
from pathlib import Path

# Option A: Clone DressApp repository directly
if not Path("DressAppV1").exists():
    !git clone https://github.com/Yoram-Jacobs/DressAppV1.git
    %cd DressAppV1

# Option B (Optional): Mount Google Drive to persist checkpoints
# from google.colab import drive
# drive.mount('/content/drive')

## 3. Configuration & Workflow Selection

In [ ]:
# Select workflow adapter
ADAPTER_NAME = "garment_vision"  # @param ["garment_vision", "trend_scout", "stylist_chat", "suitcase", "scheduled_outfit"]
BASE_MODEL = "google/gemma-4-E4B-it"  # @param {type:"string"}
DATASET_PATH = "training/datasets/sample_garment_vision.jsonl"  # @param {type:"string"}
OUTPUT_DIR = f"adapters/{ADAPTER_NAME}"

# Hyperparameters
NUM_EPOCHS = 3  # @param {type:"integer"}
BATCH_SIZE = 2  # @param {type:"integer"}
LEARNING_RATE = 2e-4  # @param {type:"number"}
MAX_SEQ_LENGTH = 2048  # @param {type:"integer"}
LORA_R = 16  # @param {type:"integer"}
LORA_ALPHA = 32  # @param {type:"integer"}

# Hugging Face Credentials
HF_TOKEN = ""  # @param {type:"string"}
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print(f"Target Adapter: {ADAPTER_NAME}")
print(f"Base Model:     {BASE_MODEL}")
print(f"Dataset:        {DATASET_PATH}")

## 4. Dataset Verification & Inspection

In [ ]:
import json

dataset_file = Path(DATASET_PATH)
assert dataset_file.exists(), f"Dataset file not found at: {dataset_file}"

records = []
with open(dataset_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print(f"Successfully loaded {len(records)} training conversation records.")
print("\n--- Sample Record Inspection ---")
print(json.dumps(records[0], indent=2))

## 5. Model Loading (4-bit NF4) & LoRA Configuration

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit NF4 Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, token=HF_TOKEN or None)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load Quantized Base Model
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN or None,
)
model = prepare_model_for_kbit_training(model)

# Configure LoRA targeting all attention and MLP projections
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Assistant-Only Loss Masking & Tokenization

In [ ]:
from datasets import Dataset

processed_samples = []
for item in records:
    messages = item["messages"]
    user_msgs = [m for m in messages if m.get("role") in ("user", "system")]
    assistant_msgs = [m for m in messages if m.get("role") == "assistant"]

    if not assistant_msgs:
        continue

    # Apply chat template
    prompt_str = tokenizer.apply_chat_template(user_msgs, tokenize=False, add_generation_prompt=True)
    full_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

    prompt_ids = tokenizer.encode(prompt_str, add_special_tokens=False)
    full_ids = tokenizer.encode(full_str, add_special_tokens=False)

    if len(full_ids) > MAX_SEQ_LENGTH:
        full_ids = full_ids[:MAX_SEQ_LENGTH]

    # Assistant-only loss masking: set all prompt tokens to -100
    prompt_len = min(len(prompt_ids), len(full_ids))
    labels = [-100] * prompt_len + full_ids[prompt_len:]

    processed_samples.append({
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    })

train_dataset = Dataset.from_list(processed_samples)
print(f"Prepared {len(train_dataset)} examples with assistant-only loss masking.")

## 7. Fine-Tuning Execution

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    warmup_ratio=0.03,
    learning_rate=LEARNING_RATE,
    fp16=False,
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    save_strategy="epoch",
    evaluation_strategy="no",
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    tokenizer=tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
)

print(f"Starting QLoRA training for {ADAPTER_NAME}...")
train_result = trainer.train()
print("Training Complete! Final Loss:", train_result.training_loss)

## 8. Export Adapter Weights & Artifacts

In [ ]:
# Save trained LoRA adapter
out_path = Path(OUTPUT_DIR)
out_path.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(out_path))
tokenizer.save_pretrained(str(out_path))
print(f"Saved adapter artifacts to {out_path}:")
for f in out_path.iterdir():
    print(f" - {f.name} ({f.stat().st_size / 1024:.1f} KB)")

## 9. Direct Upload to Hugging Face or Zip Download

In [ ]:
# Option A: Publish to Hugging Face Hub
if HF_TOKEN:
    hub_id = f"Yoram-Jacobs/dressapp-{ADAPTER_NAME}-adapter"
    print(f"Pushing adapter to {hub_id}...")
    trainer.model.push_to_hub(hub_id, token=HF_TOKEN)
    tokenizer.push_to_hub(hub_id, token=HF_TOKEN)
    print(f"Adapter published to: https://huggingface.co/{hub_id}")

# Option B: Create zip archive for local download
import shutil
zip_filename = f"dressapp_{ADAPTER_NAME}_adapter"
shutil.make_archive(zip_filename, 'zip', OUTPUT_DIR)
print(f"Created archive: {zip_filename}.zip")

try:
    from google.colab import files
    files.download(f"{zip_filename}.zip")
except Exception:
    print("Download archive manually from files sidebar.")